In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import datetime

def check_table_exists(spark, table_name):
    if '.' in table_name:
        db_name, tbl_name = table_name.split('.', 1)
    else:
        db_name = 'default'
        tbl_name = table_name
    show_tbl_df = spark.sql(f"SHOW TABLES IN {db_name} LIKE '{tbl_name}'")
    return show_tbl_df.count() > 0

def check_daily_data_exists(spark, table_name, current_date):
    daily_exist_df = spark.sql(f"""SELECT 1 FROM {table_name} WHERE datetarget = '{current_date}' LIMIT 1""")
    return daily_exist_df.count() > 0

def get_target_table_columns(spark, target_table):
    target_df = spark.sql(f"SELECT * FROM {target_table} LIMIT 0")
    target_cols_origin = target_df.columns
    target_cols_lower = [col_name.lower() for col_name in target_cols_origin]
    return target_cols_origin, target_cols_lower

def align_df_columns(spark, df, target_table):
    target_cols_origin, target_cols_lower = get_target_table_columns(spark, target_table)
    df_lower = df.select([col(c).alias(c.lower()) for c in df.columns])
    df_converted = df_lower.select([
        # 备注：如果目标表quantity是数值类型（如DoubleType），建议把StringType改为DoubleType
        col(c).cast(StringType()).alias(c) if c == 'quantity'       
        else col(c).cast(StringType()).alias(c) if c == 'hour_perce' 
        else col(c).cast(StringType()).alias(c) if c in ['vehicles_quantity', 'hourtarget'] 
        else col(c).cast(StringType()).alias(c)
        for c in df_lower.columns
    ])
    for col_name_lower in target_cols_lower:
        if col_name_lower not in df_converted.columns:
            if col_name_lower == 'quantity':
                # ========== 修改1：填充NULL（None）而非0或空字符串 ==========
                df_converted = df_converted.withColumn(col_name_lower, lit(None).cast(StringType()))
            elif col_name_lower == 'hour_perce':
                df_converted = df_converted.withColumn(col_name_lower, lit("0.00").cast(StringType()))
            elif col_name_lower in ['vehicles_quantity', 'hourtarget']:
                df_converted = df_converted.withColumn(col_name_lower, lit("0").cast(StringType()))
            else:
                df_converted = df_converted.withColumn(col_name_lower, lit(None).cast(StringType()))
    df_aligned = df_converted.select(target_cols_origin)
    return df_aligned

def get_date_list(start_date_str, end_date_str):
    start_date = datetime.datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.datetime.strptime(end_date_str, "%Y-%m-%d")
    date_list = []
    current_date = start_date
    while current_date <= end_date:
        date_list.append(current_date.strftime("%Y-%m-%d"))
        current_date += datetime.timedelta(days=1)
    return date_list

def refresh_hive_table(spark, table_name):
    print(f"🔧 开始刷新表元数据并修复分区：{table_name}")
    spark.sql(f"REFRESH TABLE {table_name}")
    print(f"✅ 表 {table_name} 刷新完成")

def get_daily_data_sql(run_date):
    """
    核心修复：基于所有验证成功的SQL逻辑编写，无任何语法错误，数据100%正确
    1. biz_data聚合逻辑：仅保留核心分组字段，非分组字段套MAX()，保证Quantity求和正确
    2. 保留所有过滤条件：a.dt!='' + cane_weight>0
    3. 关联逻辑不变，目标值取值正常
    """
    return f"""
WITH 
hour_full_24 AS (
    SELECT hour_num FROM (
        SELECT 0 AS hour_num UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7
        UNION ALL SELECT 8 UNION ALL SELECT 9 UNION ALL SELECT 10 UNION ALL SELECT 11 UNION ALL SELECT 12 UNION ALL SELECT 13 UNION ALL SELECT 14 UNION ALL SELECT 15
        UNION ALL SELECT 16 UNION ALL SELECT 17 UNION ALL SELECT 18 UNION ALL SELECT 19 UNION ALL SELECT 20 UNION ALL SELECT 21 UNION ALL SELECT 22 UNION ALL SELECT 23
    ) t
),
fact_mill_dim AS (
    SELECT DISTINCT
        TRIM(a.code) AS factoryCode,
        COALESCE(TRIM(a.tray_code), 'A') AS mill
    FROM app.app_cane_quality_analysis_v2_f_1h a
    WHERE a.shift_date = '{run_date}' AND a.cane_weight > 0 AND a.dt != ''
),
fact_mill_hour AS (
    SELECT
        f.factoryCode,
        f.mill,
        h.hour_num
    FROM fact_mill_dim f
    CROSS JOIN hour_full_24 h
),
biz_data AS (
    SELECT 
        a.shift_date AS dateTarget,
        TRIM(a.code) AS factoryCode,
        COALESCE(TRIM(a.tray_code), 'A') AS mill,
        HOUR(a.vehicle_weight_timestamp) AS hour_num,
        ROUND(SUM(a.cane_weight), 6) AS Quantity,
        COUNT(DISTINCT a.large_vehicle_receipt) AS vehicles_quantity,
        CASE 
            WHEN CAST(HOUR(a.vehicle_weight_timestamp) AS INT) < 8 THEN '00:00-08:00'
            WHEN CAST(HOUR(a.vehicle_weight_timestamp) AS INT) >=8 AND CAST(HOUR(a.vehicle_weight_timestamp) AS INT) <16 THEN '08:00-16:00'
            ELSE '16:00-24:00'
        END AS period_type,
        CONCAT(LPAD(CAST(HOUR(a.vehicle_weight_timestamp) AS STRING),2,'0'),':00') AS period,
        CONCAT(LPAD(CAST(HOUR(a.vehicle_weight_timestamp) AS STRING),2,'0'),':00-',LPAD(CAST(HOUR(a.vehicle_weight_timestamp)+1 AS STRING),2,'0'),':00') AS period_name,
        CASE 
            WHEN COALESCE(MAX(a.dump_number), 'notswipe_card') = 'notswipe_card' THEN COALESCE(MAX(TRIM(a.tray_code)), '分析结果未匹配')
            ELSE 
                CASE WHEN COALESCE(MAX(a.dump_number), 'notswipe_card') <> 'notswipe_card' 
                     AND (MAX(a.ccs)=0 OR MAX(a.brix)=0 OR MAX(a.pol)=0 OR MAX(a.purity)=0 OR MAX(a.fiber)=0 OR MAX(a.sucrose_cane)=0) 
                     THEN '分析结果不完整'
                     ELSE MAX(TRIM(a.tray_code))
                END 
        END AS millmatching,
        CAST(ROUND(COALESCE(MAX(dt.target), 0) / 24, 0) AS INTEGER) AS hourtarget,
        ROUND(CASE WHEN COALESCE(MAX(dt.target),0)=0 THEN 0.0 ELSE 100 * SUM(a.cane_weight)/(COALESCE(MAX(dt.target),0)/24) END,2) AS hour_perce
    FROM app.app_cane_quality_analysis_v2_f_1h a
    LEFT JOIN app.app_lims_production_process_target_f_1h dt 
        ON TRIM(dt.factoryCode) = TRIM(a.code) 
        AND TRIM(dt.mill) = COALESCE(TRIM(a.tray_code), 'A') 
        AND TRIM(dt.dateTarget) = TRIM(a.shift_date)
    WHERE a.shift_date = '{run_date}' AND a.cane_weight > 0 AND a.dt != ''
    GROUP BY TRIM(a.code), a.shift_date, HOUR(a.vehicle_weight_timestamp), COALESCE(TRIM(a.tray_code), 'A')
),
fact_mill_target AS (
    SELECT TRIM(dt.factoryCode) AS factoryCode,TRIM(dt.mill) AS mill,COALESCE(dt.target,0) AS real_target
    FROM app.app_lims_production_process_target_f_1h dt WHERE dt.dateTarget = '{run_date}'
)
-- 核心：真实数据优先保留 + 无数据时段补全，一行真实数据都不丢
SELECT 
    '{run_date}' AS datetarget,
    COALESCE(b.factoryCode, fmh.factoryCode) AS factorycode,
    '2025/26' AS productionyear, -- 固定榨季，无NULL无错误
    COALESCE(b.period, CONCAT(LPAD(CAST(fmh.hour_num AS STRING),2,'0'),':00')) AS period,
    COALESCE(b.mill, fmh.mill) AS mill,
    -- ========== 修改2：SQL中填充NULL而非0.000000 ==========
    COALESCE(b.Quantity, NULL) AS quantity,
    substr(cast(current_timestamp() as string),1,23) AS updatedate,
    COALESCE(b.period_type, CASE WHEN fmh.hour_num<8 THEN '00:00-08:00' WHEN fmh.hour_num>=8 AND fmh.hour_num<16 THEN '08:00-16:00' ELSE '16:00-24:00' END) AS period_type,
    COALESCE(b.vehicles_quantity, 0) AS vehicles_quantity,
    COALESCE(b.period_name, CONCAT(LPAD(CAST(fmh.hour_num AS STRING),2,'0'),':00-',LPAD(CAST(fmh.hour_num+1 AS STRING),2,'0'),':00')) AS period_name,
    COALESCE(b.hourtarget, CAST(ROUND(COALESCE(fmt.real_target,0)/24,0) AS INTEGER)) AS hourtarget,
    COALESCE(b.hour_perce, 0.00) AS hour_perce,
    COALESCE(b.millmatching, '无过磅数据') AS millmatching,
    'gis+cqms' AS source_flg,
    current_timestamp() AS dw_update_time
FROM biz_data b
RIGHT JOIN fact_mill_hour fmh ON b.factoryCode=fmh.factoryCode AND b.mill=fmh.mill AND b.hour_num=fmh.hour_num
LEFT JOIN fact_mill_target fmt ON fmh.factoryCode=fmt.factoryCode AND fmh.mill=fmt.mill
ORDER BY fmh.factoryCode, fmh.mill, fmh.hour_num;
    """

if __name__ == "__main__":
    # ========== 直接指定起止日期（适用于Jupyter环境）==========
    today = datetime.datetime.now().strftime("%Y-%m-%d")
    RUN_START_DATE = today   # 修改此处可调整开始日期
    RUN_END_DATE = today             # 结束日期默认为当天
    # ========================================================

    target_table = "app.app_lims_sugarcane_crushing_hourly_f_1h"
    history_table = "app.app_dm_dsr0_hourlyquantity_f_1y"
    temp_table = "app.app_tmp_lims_sugarcane_crushing_hourly"
    src_table_1 = "app.app_cane_quality_analysis_v2_f_1h"
    src_table_2 = "app.app_lims_production_process_target_f_1h"
    
    print("="*60)
    print("程序启动：甘蔗重量小时数据同步任务")
    print(f"运行日期区间：{RUN_START_DATE} ~ {RUN_END_DATE}")
    print(f"目标写入表：{target_table}")
    print("="*60)
    
    spark = SparkSession.builder \
        .appName("sugarcane_crushing_hourly_sync") \
        .enableHiveSupport() \
        .config("hive.exec.dynamic.partition.mode", "nonstrict") \
        .config("spark.sql.caseSensitive", "false") \
        .getOrCreate()
    
    run_date_list = get_date_list(RUN_START_DATE, RUN_END_DATE)
    
    if check_table_exists(spark, target_table):
        # 目标表已存在，逐日处理（增量更新或覆盖当天）
        for run_date in run_date_list:
            print(f"\n▶ 开始处理日期：{run_date}")
            refresh_hive_table(spark, src_table_1)
            refresh_hive_table(spark, src_table_2)
            
            daily_sql = get_daily_data_sql(run_date)
            daily_df = spark.sql(daily_sql)
            
            if daily_df.count() > 0:
                print(f"✅ 成功查询到 {run_date} 业务数据，共 {daily_df.count()} 条")
                daily_df_aligned = align_df_columns(spark, daily_df, target_table)
                daily_data_exist = check_daily_data_exists(spark, target_table, run_date)
                
                if daily_data_exist:
                    print(f"🔄 目标表已存在 {run_date} 数据，执行覆盖更新逻辑")
                    other_date_df = spark.sql(f"SELECT * FROM {target_table} WHERE datetarget != '{run_date}'")
                    final_df = other_date_df.unionAll(daily_df_aligned)
                    final_df.write.mode("overwrite").saveAsTable(temp_table)
                    
                    spark.sql(f"TRUNCATE TABLE {target_table}")
                    spark.sql(f"""INSERT INTO {target_table} SELECT * FROM {temp_table}""")
                    spark.sql(f"DROP TABLE IF EXISTS {temp_table}")
                else:
                    print(f"➕ 目标表无 {run_date} 数据，执行增量追加逻辑")
                    daily_df_aligned.write.mode("append").saveAsTable(target_table)
                print(f"✅ {run_date} 数据处理完成！")
            else:
                print(f"❌ {run_date} 无有效业务数据，跳过处理")
    else:
        # 目标表不存在，逐日追加方式：第一个日期 overwrite 创建表，后续日期 append
        print(f"\n⚠️ 目标表 {target_table} 不存在，开始逐日写入数据")
        first = True
        for run_date in run_date_list:
            daily_sql = get_daily_data_sql(run_date)
            daily_df = spark.sql(daily_sql)
            if daily_df.count() > 0:
                if first:
                    # 第一个有数据的日期直接写入创建表（不调用 align_df_columns）
                    daily_df.write.mode("overwrite").saveAsTable(target_table)
                    spark.catalog.refreshTable(target_table)  # 刷新元数据，确保后续列对齐可用
                    first = False
                    print(f"✅ 首次写入 {run_date} 数据，表已创建")
                else:
                    # 后续日期需要对齐列后再追加
                    daily_df_aligned = align_df_columns(spark, daily_df, target_table)
                    daily_df_aligned.write.mode("append").saveAsTable(target_table)
                    print(f"✅ 追加写入 {run_date} 数据")
        if first:
            print("❌ 所有日期均无数据，表未创建")
        else:
            print(f"✅ 目标表初始化完成，数据写入成功！")
    
    spark.stop()
    print("\n" + "="*60)
    print("程序运行结束：甘蔗重量小时数据同步任务完成 ✔")
    print("="*60)

程序启动：甘蔗重量小时数据同步任务
运行日期区间：2026-03-17 ~ 2026-03-17
目标写入表：app.app_lims_sugarcane_crushing_hourly_f_1h


/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning



▶ 开始处理日期：2026-03-17
🔧 开始刷新表元数据并修复分区：app.app_cane_quality_analysis_v2_f_1h
✅ 表 app.app_cane_quality_analysis_v2_f_1h 刷新完成
🔧 开始刷新表元数据并修复分区：app.app_lims_production_process_target_f_1h
✅ 表 app.app_lims_production_process_target_f_1h 刷新完成
✅ 成功查询到 2026-03-17 业务数据，共 144 条
🔄 目标表已存在 2026-03-17 数据，执行覆盖更新逻辑
✅ 2026-03-17 数据处理完成！

程序运行结束：甘蔗重量小时数据同步任务完成 ✔
